In [1]:
!pip install yfinance

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import yfinance as yf
import pandas as pd 

In [3]:
import yfinance as yf
import pandas as pd

# Step 1: Define a list of top Indian company tickers (NSE/BSE)
# You can expand this list to 1000 using NSE/BSE screener exports
tickers = [
    "INFY.BS",  # Infosys
]

# Step 2: Define a function to fetch balance sheet and working capital metrics
def fetch_financials(ticker):
    try:
        stock = yf.Ticker(ticker)
        bs = stock.balance_sheet
        wc = {}
        if not bs.empty:
            wc["Company"] = ticker
            wc["Total Current Assets"] = bs.loc["Total Current Assets"][0]
            wc["Total Current Liabilities"] = bs.loc["Total Current Liabilities"][0]
            wc["Working Capital"] = wc["Total Current Assets"] - wc["Total Current Liabilities"]
        return wc
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None

# Step 3: Loop through tickers and collect data
financial_data = []
for ticker in tickers:
    data = fetch_financials(ticker)
    if data:
        financial_data.append(data)

# Step 4: Convert to DataFrame
df = pd.DataFrame(financial_data)
print(df)

# Step 5: Save to CSV (optional)
df.to_csv("working_capital_indian_public_companies.csv", index=False)

Empty DataFrame
Columns: []
Index: []


In [4]:
import yfinance as yf
import pandas as pd

# Step 1: Define a list of top Indian company tickers (NSE/BSE)
# You can expand this list to 1000 using NSE/BSE screener exports
tickers = [
    "PACS",  # Infosys
]

# Step 2: Define a function to fetch balance sheet and working capital metrics
def fetch_financials(ticker):
    try:
        stock = yf.Ticker(ticker)
        bs = stock.balance_sheet
        wc = {}
        if not bs.empty:
            wc["Company"] = ticker
            wc["Total Current Assets"] = bs.loc["Total Current Assets"][0]
            wc["Total Current Liabilities"] = bs.loc["Total Current Liabilities"][0]
            wc["Working Capital"] = wc["Total Current Assets"] - wc["Total Current Liabilities"]
        return wc
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None

# Step 3: Loop through tickers and collect data
financial_data = []
for ticker in tickers:
    data = fetch_financials(ticker)
    if data:
        financial_data.append(data)

# Step 4: Convert to DataFrame
df = pd.DataFrame(financial_data)
print(df)

# Step 5: Save to CSV (optional)
df.to_csv("working_capital_indian_public_companies.csv", index=False)

Error fetching PACS: 'Total Current Assets'
Empty DataFrame
Columns: []
Index: []


In [11]:
import yfinance as yf
dat = yf.Ticker("INFY.NS")

In [12]:
dat

yfinance.Ticker object <INFY.NS>

In [13]:
dat.balance_sheet

,2025-03-31,2024-03-31,2023-03-31,2022-03-31,2021-03-31
Treasury Shares Number,9.655927e+06,1.091683e+07,1.217212e+07,1.372571e+07,NaN
Ordinary Shares Number,4.143608e+09,4.139951e+09,4.136388e+09,4.193013e+09,NaN
Share Issued,4.153263e+09,4.150867e+09,4.148560e+09,4.206739e+09,NaN
Total Debt,9.620000e+08,1.002000e+09,1.010000e+09,7.220000e+08,NaN
Tangible Book Value,9.700000e+09,9.517000e+09,8.077000e+09,8.899000e+09,NaN
...,...,...,...,...,...
Cash Cash Equivalents And Short Term Investments,4.509000e+09,3.433000e+09,2.359000e+09,3.229000e+09,NaN
Other Short Term Investments,1.648000e+09,1.660000e+09,8.780000e+08,9.240000e+08,NaN
Cash And Cash Equivalents,2.861000e+09,1.773000e+09,1.481000e+09,2.305000e+09,NaN
Cash Equivalents,NaN,0.000000e+00,2.610000e+08,4.650000e+08,635000000.0


In [15]:
from math import isnan

# Analyze working capital and simple liquidity metrics for available tickers
# Relies on existing imports (yfinance as yf, pandas as pd) and existing variables like `dat`, `tickers`


def _pick(bs, candidates):
    for c in candidates:
        if c in bs.index:
            try:
                val = bs.loc[c].iat[0]
                if pd.isna(val):
                    continue
                return val
            except Exception:
                continue
    return None

def analyze_tickers(ticker_list):
    results = []
    for tk in ticker_list:
        # prefer existing `dat` object if it matches
        try:
            if 'dat' in globals() and getattr(dat, "ticker", None) == tk:
                t = dat
            else:
                # try using as-is; add .NS if no exchange suffix present (best-effort)
                t = yf.Ticker(tk if '.' in tk else f"{tk}.NS")
        except Exception as e:
            print(f"Could not construct ticker for {tk}: {e}")
            continue

        try:
            bs = t.balance_sheet
            if bs is None or bs.empty:
                print(f"No balance sheet for {tk}")
                continue

            c_assets = _pick(bs, ["Total Current Assets", "Total current assets", "TotalCurrentAssets", "Current Assets"])
            c_liab  = _pick(bs, ["Total Current Liabilities", "Total current liabilities", "TotalCurrentLiabilities", "Current Liabilities"])
            inventory = _pick(bs, ["Inventory", "Inventories", "Total Inventory"])

            # convert to numeric or None
            def num(x):
                try:
                    if x is None:
                        return None
                    if isinstance(x, (int, float)):
                        if isnan(x):
                            return None
                        return float(x)
                    return float(x)
                except Exception:
                    return None

            ca = num(c_assets); cl = num(c_liab); inv = num(inventory)

            wc = None if (ca is None or cl is None) else (ca - cl)
            current_ratio = None if (ca is None or cl is None or cl == 0) else (ca / cl)
            quick_ratio = None
            if inv is not None and ca is not None and cl is not None and cl != 0:
                quick_ratio = (ca - inv) / cl

            info = {}
            try:
                info = t.info or {}
            except Exception:
                info = {}

            results.append({
                "Company": tk,
                "Latest_Current_Assets": ca,
                "Latest_Current_Liabilities": cl,
                "Inventory": inv,
                "Working Capital": wc,
                "Current Ratio": current_ratio,
                "Quick Ratio": quick_ratio,
                "MarketCap": info.get("marketCap")
            })
        except Exception as e:
            print(f"Error processing {tk}: {e}")
            continue

    return results

# build the list to analyze: use existing tickers plus the `dat` ticker if available
to_analyze = list(tickers)  # uses existing variable
if 'dat' in globals():
    dt = getattr(dat, "ticker", None)
    if dt and dt not in to_analyze:
        to_analyze.append(dt)

financial_data = analyze_tickers(to_analyze)

# create/update dataframe `df`
df = pd.DataFrame(financial_data)
display(df)

# quick insights
if not df.empty:
    neg_wc = df[df["Working Capital"] < 0].Company.tolist() if "Working Capital" in df.columns else []
    low_current = df[df["Current Ratio"] < 1].Company.tolist() if "Current Ratio" in df.columns else []
    print("Companies with negative working capital:", neg_wc)
    print("Companies with current ratio < 1:", low_current)
    print("Top 5 by MarketCap (if available):")
    if "MarketCap" in df.columns:
        display(df.sort_values("MarketCap", ascending=False).head(5)[["Company", "MarketCap", "Working Capital", "Current Ratio"]])
else:
    print("No financial data collected. Try adding valid NSE tickers (e.g., 'INFY.NS') to `tickers` list or verify yfinance connectivity.")

No balance sheet for PACS


,Company,Latest_Current_Assets,Latest_Current_Liabilities,Inventory,Working Capital,Current Ratio,Quick Ratio,MarketCap
0,INFY.NS,1.135900e+10,5.012000e+09,None,6.347000e+09,2.266361,None,6375072071680


Companies with negative working capital: []
Companies with current ratio < 1: []
Top 5 by MarketCap (if available):


,Company,MarketCap,Working Capital,Current Ratio
0,INFY.NS,6375072071680,6.347000e+09,2.266361


SyntaxError: invalid syntax (2120762996.py, line 1)